# Colab: **RAG module** (FAISS + sentence-transformers)

This notebook exercises **`rag/retriever.py`**:

- Builds or loads **`index.faiss`** + **`meta.json`** under **`RAG_INDEX_DIR`** (default: project **`rag_index/`**).
- Corpus: **`rag/sample_corpus.txt`** unless you set **`RAG_CORPUS_PATH`** to your own `.txt` file.
- Calls **`FaissRetriever.retrieve_context(query, k)`** → top‑k snippet strings fed into the worker’s LLM path.

Use this to **inspect retrieval quality**, **swap corpora**, or **rebuild the index** before running **`01_GPU_Worker.ipynb`** with the **same** `RAG_INDEX_DIR` (persist `rag_index/` in Drive or bake into repo).

## 1) Project root

In [ ]:
import os

# !git clone https://github.com/YOUR_ORG/YOUR_REPO.git /content/dist

PROJECT_ROOT = "/content/dist"
os.chdir(PROJECT_ROOT)
print("cwd:", os.getcwd())
assert os.path.isdir("rag"), "Fix PROJECT_ROOT (need rag/)"

## 2) Install RAG deps

**`requirements-rag.txt`** (`faiss-cpu`, `sentence-transformers`, `numpy`). Colab already includes **torch**; `sentence-transformers` uses it.

In [ ]:
%pip install -q -r requirements-rag.txt

## 3) Paths and embedding model

- **`RAG_INDEX_DIR`**: where FAISS index + chunk JSON live (`/content/rag_index_keep` + Drive sync is convenient).
- **`RAG_EMBED_MODEL`**: defaults to **`sentence-transformers/all-MiniLM-L6-v2`**.
- **`RAG_CORPUS_PATH`**: optional full path to a ** UTF-8 `.txt`** file used **only when** the index is **missing** and must be **built**. Leave unset to use built-in **`rag/sample_corpus.txt`**.

In [ ]:
import os

os.chdir(PROJECT_ROOT)

os.environ.setdefault("RAG_INDEX_DIR", os.path.join(PROJECT_ROOT, "rag_index"))
os.environ.setdefault("RAG_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")

# Optional custom corpus path (unset = use rag/sample_corpus.txt)
# os.environ["RAG_CORPUS_PATH"] = os.path.join(PROJECT_ROOT, "rag/my_docs.txt")

print("RAG_INDEX_DIR:", os.environ["RAG_INDEX_DIR"])
print("RAG_EMBED_MODEL:", os.environ["RAG_EMBED_MODEL"])
print("RAG_CORPUS_PATH:", os.environ.get("RAG_CORPUS_PATH", "(default rag/sample_corpus.txt)"))

## 4) (Optional) Custom corpus snippet

Writes **`rag/colab_upload_corpus.txt`** and points **`RAG_CORPUS_PATH`** at it.

**Skip** if you rely on **`rag/sample_corpus.txt`**.

In [ ]:
import os

CUSTOM = """\
Colab notebooks can preprocess RAG indexes before deploying workers.
FAISS Inner Product with normalized vectors approximates cosine similarity.
Workers read RAG_INDEX_DIR from the environment on each task.
"""

os.chdir(PROJECT_ROOT)
path = os.path.join(PROJECT_ROOT, "rag", "colab_upload_corpus.txt")
os.makedirs(os.path.dirname(path), exist_ok=True)
with open(path, "w", encoding="utf-8") as f:
    f.write(CUSTOM)

os.environ["RAG_CORPUS_PATH"] = path
print("Wrote:", path)

## 5) (Optional) Force rebuild

Delete existing index so **`ensure_loaded()`** rebuilds from the corpus (needed after changing **`RAG_CORPUS_PATH`** or chunking logic).

In [ ]:
import os
import shutil

idx_dir = os.environ["RAG_INDEX_DIR"]
if os.path.isdir(idx_dir):
    shutil.rmtree(idx_dir)
    print("Removed:", idx_dir)
else:
    print("No index dir yet:", idx_dir)

## 6) Build / load index and run queries

First build downloads the embedding model weights (one-time).

In [ ]:
import os
import sys

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from rag.retriever import FaissRetriever, retrieve_context

r = FaissRetriever()
r.ensure_loaded()

print("index_dir:", r.index_dir)
print("chunks:", len(r._chunks))
print()

for q in [
    "What is FAISS?",
    "How do workers use RAG?",
    "Redis Streams consumer groups",
]:
    ctx = r.retrieve_context(q, k=3)
    print("Q:", q)
    for i, c in enumerate(ctx, 1):
        print(f"  [{i}] {c[:120]}{'...' if len(c) > 120 else ''}")
    print()

# Module-level helper (same env)
print("retrieve_context():", retrieve_context("load balancing", k=2))

## 7) Next steps

- Point **`01_GPU_Worker.ipynb`** at the **same** **`RAG_INDEX_DIR`** (or copy `rag_index/` into the worker runtime).
- Control plane does **not** run RAG; only **workers** call **`retrieve_context`** during inference.